# Vendor Risk Segmentation Report

This notebook profiles vendors by **operational risk** using the **VRIS score** as the segmentation variable.

## Deliverables Covered

- risk-category creation: `Low Risk`, `Medium Risk`, `High Risk`, `Critical Risk`
- category-level analysis of:
  - procurement spend
  - average delivery delay
  - product quality
  - lead time
  - revenue dependency
- visual report outputs
- category risk profiles
- category-specific recommendations

## Modeling Choice

The vendor dataset is a **monthly snapshot** table. To avoid mixing past and current states into one profile, the analysis uses:

1. the **latest available snapshot per vendor** for current-state risk features;
2. a **trailing-12-month procurement spend** view for spend scale;
3. vendor-level order aggregates for delivery delay and revenue dependency.

This gives us a practical current-risk segmentation rather than a blended 48-month history.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (12, 7)

ROOT = Path.cwd().resolve()
DATA_DIR = ROOT / "data"

print(f"Working directory: {ROOT}")
print(f"Data directory: {DATA_DIR}")

## Load The Datasets Used In This Report

The main segmentation comes from `vendors.csv`, while `orders.csv` is used to enrich the vendor profiles with:

- delivery delay
- vendor-linked revenue
- revenue dependency share

`financials.csv` is not required for the core report because `orders.csv` already provides consistent vendor-linked revenue coverage across far more vendor IDs.

In [ ]:
vendors = pd.read_csv(DATA_DIR / "vendors.csv")
orders = pd.read_csv(DATA_DIR / "orders.csv")

print("vendors shape:", vendors.shape)
print("orders shape:", orders.shape)

In [ ]:
def normalize_object_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Trim whitespace from text fields without changing missing values."""
    object_columns = df.select_dtypes(include="object").columns
    for column in object_columns:
        df[column] = df[column].map(lambda value: value.strip() if isinstance(value, str) else value)
    return df


def segment_from_quantiles(series: pd.Series) -> tuple[pd.Series, dict]:
    """Create four balanced risk groups from the VRIS distribution.

    Percentile-based cutoffs are used here because the observed VRIS distribution
    is concentrated in the upper half of the 0-100 scale. Fixed business bands
    like 0-25-50-75 would place almost every vendor into High or Critical risk.
    Quantile segmentation keeps the categories analytically useful.
    """
    q1 = series.quantile(0.25)
    q2 = series.quantile(0.50)
    q3 = series.quantile(0.75)

    segmented = pd.cut(
        series,
        bins=[-np.inf, q1, q2, q3, np.inf],
        labels=["Low Risk", "Medium Risk", "High Risk", "Critical Risk"],
        include_lowest=True,
    )

    thresholds = {"q1": float(q1), "q2": float(q2), "q3": float(q3)}
    return segmented, thresholds


def latest_snapshot_per_vendor(vendors_df: pd.DataFrame) -> pd.DataFrame:
    """Keep the latest monthly vendor record for each vendor_id."""
    latest = (
        vendors_df.sort_values(["vendor_id", "snapshot_month"])
        .groupby("vendor_id", as_index=False)
        .tail(1)
        .copy()
    )
    return latest


def display_top_rows(title: str, df: pd.DataFrame, n: int = 5) -> None:
    print(title)
    display(df.head(n))

## Prepare Vendors Data

This step does **light analysis-focused preparation**, not a full data-cleaning rebuild.

The main goals are:

- parse dates;
- standardize text fields;
- create a usable VRIS score for every vendor;
- create a stable vendor-level spend view.

In [ ]:
vendors_prepared = vendors.copy()
vendors_prepared = normalize_object_columns(vendors_prepared)

for column in ["vendor_record_id", "vendor_id"]:
    vendors_prepared[column] = vendors_prepared[column].str.upper()

vendors_prepared["snapshot_month"] = pd.to_datetime(vendors_prepared["snapshot_month"], errors="coerce")
vendors_prepared["contract_start_date"] = pd.to_datetime(vendors_prepared["contract_start_date"], errors="coerce")
vendors_prepared["contract_expiry_date"] = pd.to_datetime(vendors_prepared["contract_expiry_date"], errors="coerce")

vendors_prepared["vendor_tier"] = vendors_prepared["vendor_tier"].str.title()
vendors_prepared["vendor_category"] = vendors_prepared["vendor_category"].str.title()

# Some vendors have missing VRIS scores, but the report needs a segmentation variable for every vendor.
# We use a transparent proxy driven by the same operational risk signals already present in the dataset.
lead_time_risk = np.where(
    vendors_prepared["contract_lead_time_days"].gt(0),
    (vendors_prepared["lead_time_variance_days"] / vendors_prepared["contract_lead_time_days"] * 100).clip(0, 100),
    0,
)

vris_proxy = (
    0.20 * (100 - vendors_prepared["on_time_delivery_rate"].clip(0, 100))
    + 0.20 * (100 - vendors_prepared["quality_acceptance_rate"].fillna(vendors_prepared["quality_acceptance_rate"].median()).clip(0, 100))
    + 0.15 * (vendors_prepared["defect_rate_pct"].clip(0, 15) / 15 * 100)
    + 0.15 * lead_time_risk
    + 0.10 * vendors_prepared["concentration_risk_pct"].clip(0, 100)
    + 0.10 * (100 - vendors_prepared["financial_stability_score"].fillna(vendors_prepared["financial_stability_score"].median()).clip(0, 100))
    + 0.10 * (vendors_prepared["active_dispute_flag"].clip(0, 1) * 100)
).clip(0, 100)

vendors_prepared["vris_score_filled"] = vendors_prepared["vris_score"].fillna(vris_proxy.round(2))
vendors_prepared["vris_imputed_flag"] = vendors_prepared["vris_score"].isna()

latest_snapshot_date = vendors_prepared["snapshot_month"].max()
trailing_12m_start = latest_snapshot_date - pd.DateOffset(months=11)

latest_vendor_snapshot = latest_snapshot_per_vendor(vendors_prepared)

trailing_spend = (
    vendors_prepared.loc[vendors_prepared["snapshot_month"].between(trailing_12m_start, latest_snapshot_date)]
    .groupby("vendor_id", as_index=False)["procurement_spend_usd"]
    .sum()
    .rename(columns={"procurement_spend_usd": "procurement_spend_t12m_usd"})
)

display_top_rows("Latest vendor snapshot", latest_vendor_snapshot[[
    "vendor_id", "vendor_name", "snapshot_month", "vris_score", "vris_score_filled", "procurement_spend_usd"
]])

## Prepare Orders Data For Vendor KPIs

`orders.csv` is used to derive:

- average delivery delay per vendor
- total vendor-linked revenue
- revenue dependency share

For delivery analysis, only rows with an actual delivery outcome are used. This avoids mixing active or cancelled orders into a delay metric.

In [ ]:
orders_prepared = orders.copy()
orders_prepared = normalize_object_columns(orders_prepared)

for column in ["order_id", "vendor_id"]:
    orders_prepared[column] = orders_prepared[column].str.upper()

orders_prepared["order_date"] = pd.to_datetime(orders_prepared["order_date"], errors="coerce")
orders_prepared["actual_delivery_date"] = pd.to_datetime(orders_prepared["actual_delivery_date"], errors="coerce")
orders_prepared["promised_delivery_date"] = pd.to_datetime(orders_prepared["promised_delivery_date"], errors="coerce")

orders_prepared["order_status"] = orders_prepared["order_status"].str.upper()

# Recompute delivery delay from the date columns when possible because it is a derived metric.
recomputed_delay = (
    orders_prepared["actual_delivery_date"] - orders_prepared["promised_delivery_date"]
).dt.days
orders_prepared["delivery_delay_days_final"] = orders_prepared["delivery_delay_days"].fillna(recomputed_delay)

# Build vendor-level KPIs from the orders table.
delivery_orders = orders_prepared.loc[
    orders_prepared["actual_delivery_date"].notna() & orders_prepared["vendor_id"].notna()
].copy()

orders_vendor_kpis = (
    orders_prepared.groupby("vendor_id", as_index=False)
    .agg(
        total_vendor_revenue_usd=("order_value_usd", "sum"),
        total_vendor_orders=("order_id", "count"),
    )
)

delivery_vendor_kpis = (
    delivery_orders.groupby("vendor_id", as_index=False)
    .agg(
        avg_delivery_delay_days=("delivery_delay_days_final", "mean"),
        median_delivery_delay_days=("delivery_delay_days_final", "median"),
        delivered_orders=("order_id", "count"),
    )
)

orders_vendor_kpis = orders_vendor_kpis.merge(delivery_vendor_kpis, on="vendor_id", how="left")

total_revenue_linked_to_vendors = orders_vendor_kpis["total_vendor_revenue_usd"].sum()
orders_vendor_kpis["revenue_dependency_pct"] = np.where(
    total_revenue_linked_to_vendors > 0,
    orders_vendor_kpis["total_vendor_revenue_usd"] / total_revenue_linked_to_vendors * 100,
    np.nan,
)

display_top_rows("Vendor KPI sample from orders", orders_vendor_kpis)

## Build The Current Vendor Risk Profile Table

This table is the core analytical dataset for the report. It combines:

- the latest vendor snapshot;
- trailing-12-month procurement spend;
- vendor-level delivery and revenue KPIs from orders.

In [ ]:
vendor_profile = latest_vendor_snapshot.merge(trailing_spend, on="vendor_id", how="left")
vendor_profile = vendor_profile.merge(orders_vendor_kpis, on="vendor_id", how="left")

# Fill spend with the latest-month procurement spend when trailing-12-month aggregation is unavailable.
vendor_profile["procurement_spend_t12m_usd"] = vendor_profile["procurement_spend_t12m_usd"].fillna(
    vendor_profile["procurement_spend_usd"]
)

# Vendors without order linkage should remain in the population; their order-derived KPIs become zero or missing.
vendor_profile["total_vendor_revenue_usd"] = vendor_profile["total_vendor_revenue_usd"].fillna(0)
vendor_profile["total_vendor_orders"] = vendor_profile["total_vendor_orders"].fillna(0)
vendor_profile["delivered_orders"] = vendor_profile["delivered_orders"].fillna(0)

# Missing delay means the vendor has no delivered orders in the matched orders dataset.
vendor_profile["avg_delivery_delay_days"] = vendor_profile["avg_delivery_delay_days"].fillna(0)
vendor_profile["median_delivery_delay_days"] = vendor_profile["median_delivery_delay_days"].fillna(0)
vendor_profile["revenue_dependency_pct"] = vendor_profile["revenue_dependency_pct"].fillna(0)

vendor_profile["risk_category_4"], risk_thresholds = segment_from_quantiles(vendor_profile["vris_score_filled"])

vendor_profile["contract_expired_flag"] = (
    vendor_profile["contract_expiry_date"].notna()
) & (
    vendor_profile["contract_expiry_date"] < vendor_profile["snapshot_month"]
)

display(vendor_profile[[
    "vendor_id",
    "vendor_name",
    "vris_score_filled",
    "risk_category_4",
    "procurement_spend_t12m_usd",
    "avg_delivery_delay_days",
    "quality_acceptance_rate",
    "lead_time_avg_days",
    "revenue_dependency_pct",
]].head())

print("Quantile thresholds used for segmentation:", risk_thresholds)

## Category Distribution

Before analyzing the profiles, we first confirm how vendors are distributed across the four risk groups.

In [ ]:
category_counts = (
    vendor_profile["risk_category_4"]
    .value_counts()
    .rename_axis("risk_category")
    .reset_index(name="vendor_count")
)

display(category_counts)

plt.figure(figsize=(10, 6))
sns.barplot(data=category_counts, x="risk_category", y="vendor_count", palette="RdYlGn_r")
plt.title("Vendor Count By Risk Category")
plt.xlabel("Risk Category")
plt.ylabel("Number of Vendors")
plt.tight_layout()
plt.show()

## Category-Level Metrics

These are the core report metrics requested in the task:

- procurement spend
- average delivery delay
- product quality
- lead time
- revenue dependency

The table below summarizes the current vendor population by risk segment.

In [ ]:
category_summary = (
    vendor_profile.groupby("risk_category_4", observed=True)
    .agg(
        vendor_count=("vendor_id", "nunique"),
        avg_vris_score=("vris_score_filled", "mean"),
        procurement_spend_t12m_usd=("procurement_spend_t12m_usd", "mean"),
        avg_delivery_delay_days=("avg_delivery_delay_days", "mean"),
        quality_acceptance_rate=("quality_acceptance_rate", "mean"),
        defect_rate_pct=("defect_rate_pct", "mean"),
        lead_time_avg_days=("lead_time_avg_days", "mean"),
        revenue_dependency_pct=("revenue_dependency_pct", "mean"),
        contract_expired_share=("contract_expired_flag", "mean"),
        dispute_share=("active_dispute_flag", "mean"),
    )
    .reset_index()
    .rename(columns={"risk_category_4": "risk_category"})
)

display(category_summary)

## Charts For The Report

The following visuals compare the requested business metrics across the four risk categories.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

sns.barplot(data=category_summary, x="risk_category", y="procurement_spend_t12m_usd", palette="RdYlGn_r", ax=axes[0, 0])
axes[0, 0].set_title("Average Trailing-12-Month Procurement Spend")
axes[0, 0].set_xlabel("Risk Category")
axes[0, 0].set_ylabel("USD")

sns.barplot(data=category_summary, x="risk_category", y="avg_delivery_delay_days", palette="RdYlGn_r", ax=axes[0, 1])
axes[0, 1].set_title("Average Delivery Delay")
axes[0, 1].set_xlabel("Risk Category")
axes[0, 1].set_ylabel("Days")

sns.barplot(data=category_summary, x="risk_category", y="quality_acceptance_rate", palette="RdYlGn_r", ax=axes[1, 0])
axes[1, 0].set_title("Average Product Quality Acceptance Rate")
axes[1, 0].set_xlabel("Risk Category")
axes[1, 0].set_ylabel("Percent")

sns.barplot(data=category_summary, x="risk_category", y="lead_time_avg_days", palette="RdYlGn_r", ax=axes[1, 1])
axes[1, 1].set_title("Average Lead Time")
axes[1, 1].set_xlabel("Risk Category")
axes[1, 1].set_ylabel("Days")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
sns.scatterplot(
    data=vendor_profile,
    x="vris_score_filled",
    y="procurement_spend_t12m_usd",
    hue="risk_category_4",
    size="revenue_dependency_pct",
    palette="RdYlGn_r",
    sizes=(40, 400),
    alpha=0.8,
)
plt.title("VRIS Score vs Procurement Spend")
plt.xlabel("VRIS Score")
plt.ylabel("Trailing-12-Month Procurement Spend (USD)")
plt.tight_layout()
plt.show()

In [ ]:
profile_metrics = category_summary.set_index("risk_category")[
    [
        "procurement_spend_t12m_usd",
        "avg_delivery_delay_days",
        "quality_acceptance_rate",
        "lead_time_avg_days",
        "revenue_dependency_pct",
    ]
]

# Standardize the metrics to make the cross-category profile comparison more visually readable.
profile_heatmap = profile_metrics.apply(
    lambda column: (column - column.mean()) / column.std(ddof=0) if column.std(ddof=0) else column * 0,
    axis=0,
)

plt.figure(figsize=(12, 6))
sns.heatmap(profile_heatmap, annot=True, cmap="RdYlGn_r", center=0, fmt=".2f")
plt.title("Relative Risk Profile By Category (Z-Scored Metrics)")
plt.xlabel("Metric")
plt.ylabel("Risk Category")
plt.tight_layout()
plt.show()

## Risk Profiles

This section converts the category summary into a business-readable risk narrative.  
The logic below is intentionally explicit so the notebook remains easy to explain and audit.

In [ ]:
overall_metrics = {
    "procurement_spend_t12m_usd": vendor_profile["procurement_spend_t12m_usd"].mean(),
    "avg_delivery_delay_days": vendor_profile["avg_delivery_delay_days"].mean(),
    "quality_acceptance_rate": vendor_profile["quality_acceptance_rate"].mean(),
    "lead_time_avg_days": vendor_profile["lead_time_avg_days"].mean(),
    "revenue_dependency_pct": vendor_profile["revenue_dependency_pct"].mean(),
}


def compare_metric(value: float, baseline: float, higher_is_riskier: bool) -> str:
    if pd.isna(value) or pd.isna(baseline):
        return "insufficient data"
    if np.isclose(value, baseline):
        return "near portfolio average"

    if higher_is_riskier:
        return "above portfolio average" if value > baseline else "below portfolio average"
    return "below portfolio average" if value > baseline else "above portfolio average"


risk_profiles = []
for _, row in category_summary.iterrows():
    risk_profiles.append(
        {
            "risk_category": row["risk_category"],
            "risk_profile": (
                f"{row['risk_category']} vendors have average VRIS {row['avg_vris_score']:.2f}, "
                f"procurement spend {compare_metric(row['procurement_spend_t12m_usd'], overall_metrics['procurement_spend_t12m_usd'], True)}, "
                f"delivery delay {compare_metric(row['avg_delivery_delay_days'], overall_metrics['avg_delivery_delay_days'], True)}, "
                f"product quality {compare_metric(row['quality_acceptance_rate'], overall_metrics['quality_acceptance_rate'], False)}, "
                f"lead time {compare_metric(row['lead_time_avg_days'], overall_metrics['lead_time_avg_days'], True)}, "
                f"and revenue dependency {compare_metric(row['revenue_dependency_pct'], overall_metrics['revenue_dependency_pct'], True)}."
            ),
        }
    )

risk_profiles_df = pd.DataFrame(risk_profiles)
display(risk_profiles_df)

## Recommendations

Recommendations are aligned to how risk severity changes across the categories.  
These are meant to be usable directly in the final report.

In [ ]:
recommendations_df = pd.DataFrame(
    [
        {
            "risk_category": "Low Risk",
            "recommendation": "Maintain current sourcing levels, keep standard quarterly reviews, and use these vendors as benchmark partners for service and quality expectations.",
        },
        {
            "risk_category": "Medium Risk",
            "recommendation": "Monitor delivery delay and quality drift monthly, tighten SLA follow-up, and prepare secondary options for categories with rising revenue dependency.",
        },
        {
            "risk_category": "High Risk",
            "recommendation": "Launch targeted vendor improvement plans, reduce single-vendor exposure where dependency is elevated, and review contract terms, lead-time buffers, and dispute escalation cadence.",
        },
        {
            "risk_category": "Critical Risk",
            "recommendation": "Escalate to executive supplier-risk review, diversify sourcing immediately, cap new volume allocation, and require corrective action plans tied to delivery, quality, and continuity milestones.",
        },
    ]
)

display(recommendations_df)

## High-Risk Vendor Watchlist

This table highlights the vendors that merit the fastest action. It is sorted by risk severity and business exposure.

In [ ]:
watchlist = (
    vendor_profile.loc[vendor_profile["risk_category_4"].isin(["High Risk", "Critical Risk"])]
    .sort_values(["vris_score_filled", "procurement_spend_t12m_usd", "revenue_dependency_pct"], ascending=[False, False, False])
    [
        [
            "vendor_id",
            "vendor_name",
            "risk_category_4",
            "vris_score_filled",
            "procurement_spend_t12m_usd",
            "avg_delivery_delay_days",
            "quality_acceptance_rate",
            "lead_time_avg_days",
            "revenue_dependency_pct",
            "active_dispute_flag",
            "contract_expired_flag",
        ]
    ]
)

display(watchlist.head(20))

## Final Report Snapshot

The outputs below are the main report-ready tables:

- `category_summary`
- `risk_profiles_df`
- `recommendations_df`
- `watchlist`

If needed, they can be exported to CSV in the optional cell below.

In [ ]:
# OUTPUT_DIR = ROOT / "Reports" / "vendor_risk_segmentation"
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
#
# category_summary.to_csv(OUTPUT_DIR / "category_summary.csv", index=False)
# risk_profiles_df.to_csv(OUTPUT_DIR / "risk_profiles.csv", index=False)
# recommendations_df.to_csv(OUTPUT_DIR / "recommendations.csv", index=False)
# watchlist.to_csv(OUTPUT_DIR / "vendor_watchlist.csv", index=False)
#
# print(f"Report tables saved to: {OUTPUT_DIR}")